# Auto Tagging Support Tickets Using LLM

This notebook demonstrates how to automatically tag customer support tickets using Large Language Models (LLMs). We will explore both Zero-Shot Classification and a simulated Few-Shot Prompting approach using the `facebook/bart-large-mnli` model.

In [ ]:
import pandas as pd

df = pd.read_csv("customer_support_tickets.csv")
print(df.columns.tolist())
print(df["Ticket Type"].value_counts())

## 1. Data Loading and Initial Exploration

The dataset `customer_support_tickets.csv` contains customer support ticket information. We will load this data into a Pandas DataFrame and perform an initial inspection.

In [ ]:
# Display the first few rows of the DataFrame
display(df.head())

# Get a concise summary of the DataFrame
df.info()

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

## 2. Data Preprocessing

We need to ensure that the `Ticket Description` and `Ticket Type` columns are clean and suitable for our classification task. This involves handling missing values and verifying the consistency of `Ticket Type` classes.

In [ ]:
# Drop rows where 'Ticket Description' or 'Ticket Type' is missing
df.dropna(subset=['Ticket Description', 'Ticket Type'], inplace=True)

# Verify the unique 'Ticket Type' classes are as expected
expected_ticket_types = [
    'Refund request',
    'Technical issue',
    'Cancellation request',
    'Product inquiry',
    'Billing inquiry'
]

# Filter out any rows with 'Ticket Type' not in our expected list, if any exist
df = df[df['Ticket Type'].isin(expected_ticket_types)]

print(f"\nCleaned data shape: {df.shape}")
print(f"Unique 'Ticket Type' values after cleaning: {df['Ticket Type'].unique().tolist()}")

## 3. Class Distribution Visualization

Before proceeding with classification, let's visualize the distribution of our target variable, `Ticket Type`, to understand class balance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.countplot(y='Ticket Type', data=df, order=df['Ticket Type'].value_counts().index, palette='viridis')
plt.title('Distribution of Ticket Types')
plt.xlabel('Number of Tickets')
plt.ylabel('Ticket Type')
plt.tight_layout()
plt.show()

## 4. Zero-Shot Classification using `facebook/bart-large-mnli`

Zero-shot classification allows us to classify text into categories without explicit training on those categories. We will use the `facebook/bart-large-mnli` model, which is fine-tuned on Natural Language Inference (NLI) tasks, making it suitable for this purpose.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install transformers[torch] -q
from transformers import pipeline
import torch # Import torch

# Initialize the zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0 if torch.cuda.is_available() else -1)

# Define the candidate labels (our ticket types)
candidate_labels = expected_ticket_types

def zero_shot_predict(text, labels):
    result = classifier(text, labels, multi_label=False)
    # Return the predicted label and all scores in order
    return result['labels'][0], result['scores']

# Apply zero-shot classification to a sample of the data due to potential runtime for full dataset
# For demonstration, let's process a smaller sample to avoid long execution times in Colab
sample_df = df.sample(n=min(len(df), 1000), random_state=42).copy() # Process up to 1000 samples

print(f"Processing {len(sample_df)} samples for zero-shot classification...")
zero_shot_results = sample_df['Ticket Description'].apply(lambda x: zero_shot_predict(x, candidate_labels))

sample_df['zero_shot_predicted_label'] = [res[0] for res in zero_shot_results]
sample_df['zero_shot_prediction_scores'] = [res[1] for res in zero_shot_results]

display(sample_df[['Ticket Description', 'Ticket Type', 'zero_shot_predicted_label']].head())
print("Zero-shot classification complete.")

## 5. Few-Shot Prompting (Simulated) using `facebook/bart-large-mnli`

While `facebook/bart-large-mnli` is primarily a zero-shot model, we can simulate a 'few-shot' approach by concatenating manually crafted examples to the input description. This provides additional context to the model, although it's not true in-context learning as seen in generative LLMs. We expect the model to use the context from the examples to improve its classification for the target ticket.

In [ ]:
# Select a few examples for few-shot prompting
# For simplicity, we'll pick one example for each ticket type

few_shot_examples = {}
for ticket_type in expected_ticket_types:
    example = df[df['Ticket Type'] == ticket_type].iloc[0]
    few_shot_examples[ticket_type] = f"Ticket Description: {example['Ticket Description']}\nTicket Type: {example['Ticket Type']}"

# Construct the few-shot prompt template
def create_few_shot_prompt(ticket_description, examples):
    prompt = "Here are some examples of support tickets and their types:\n"
    for tt, ex_text in examples.items():
        prompt += f"- {ex_text}\n"
    prompt += f"\nNow, classify the following ticket:\nTicket Description: {ticket_description}\nTicket Type:"
    return prompt

def few_shot_predict(text, labels, examples):
    # Although we create a 'prompt', the Bart model still processes the full text against candidate labels
    # The 'few-shot' aspect here is the additional context provided by the examples in the input text.
    full_text_with_examples = create_few_shot_prompt(text, examples)
    result = classifier(full_text_with_examples, labels, multi_label=False)
    return result['labels'][0], result['scores']

print(f"Processing {len(sample_df)} samples for few-shot classification...")
few_shot_results = sample_df['Ticket Description'].apply(lambda x: few_shot_predict(x, candidate_labels, few_shot_examples))

sample_df['few_shot_predicted_label'] = [res[0] for res in few_shot_results]
sample_df['few_shot_prediction_scores'] = [res[1] for res in few_shot_results]

display(sample_df[['Ticket Description', 'Ticket Type', 'few_shot_predicted_label']].head())
print("Few-shot classification complete.")

## 6. Performance Evaluation

Now, let's evaluate the performance of both Zero-Shot and Few-Shot (simulated) approaches using standard classification metrics: Accuracy, Precision, Recall, F1 Score, and Classification Report. We will also generate Confusion Matrices.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import numpy as np

def evaluate_model(true_labels, predicted_labels, method_name):
    print(f"\n--- {method_name} Performance ---")
    accuracy = accuracy_score(true_labels, predicted_labels)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='weighted', zero_division=0)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(true_labels, predicted_labels, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(true_labels, predicted_labels, labels=candidate_labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=candidate_labels, yticklabels=candidate_labels)
    plt.title(f'Confusion Matrix - {method_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.show()

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1_score': f1}

# Evaluate Zero-Shot
zero_shot_metrics = evaluate_model(sample_df['Ticket Type'], sample_df['zero_shot_predicted_label'], "Zero-Shot Classification")

# Evaluate Few-Shot
few_shot_metrics = evaluate_model(sample_df['Ticket Type'], sample_df['few_shot_predicted_label'], "Few-Shot Classification (Simulated)")

## 7. Output Top 3 Probable Tags and Save Predictions

For each ticket, we will determine the top 3 most probable tags based on the classification scores. We will then save these predictions to a CSV file.

In [ ]:
import pandas as pd

def get_top_n_tags(prediction_scores, labels, n=3):
    # Create a dictionary of scores mapped to labels
    score_label_map = {labels[i]: score for i, score in enumerate(prediction_scores)}
    # Sort by score in descending order and get top N labels
    sorted_tags = sorted(score_label_map.items(), key=lambda item: item[1], reverse=True)
    return [tag for tag, _ in sorted_tags[:n]]

# Add Top 3 tags for Zero-Shot
sample_df['zero_shot_top_3_tags'] = sample_df.apply(
    lambda row: get_top_n_tags(row['zero_shot_prediction_scores'], candidate_labels, n=3),
    axis=1
)

# Add Top 3 tags for Few-Shot
sample_df['few_shot_top_3_tags'] = sample_df.apply(
    lambda row: get_top_n_tags(row['few_shot_prediction_scores'], candidate_labels, n=3),
    axis=1
)

# Display a sample with top 3 tags
display(sample_df[['Ticket Description', 'Ticket ID', 'Ticket Type', 'zero_shot_predicted_label', 'zero_shot_top_3_tags', 'few_shot_predicted_label', 'few_shot_top_3_tags']].head())

# Save predictions to CSV
# Include 'Ticket ID' as it's a unique identifier for each ticket
predictions_df = sample_df[['Ticket ID', 'Ticket Description', 'Ticket Type', 'zero_shot_predicted_label', 'zero_shot_top_3_tags', 'few_shot_predicted_label', 'few_shot_top_3_tags']]
predictions_df.to_csv('ticket_predictions.csv', index=False)
print("\nPredictions saved to 'ticket_predictions.csv'")

## 8. Accuracy Comparison Visualization

Let's visualize the accuracy comparison between the Zero-Shot and Few-Shot (simulated) approaches.

In [ ]:
metrics_data = {
    'Method': ['Zero-Shot', 'Few-Shot (Simulated)'],
    'Accuracy': [zero_shot_metrics['accuracy'], few_shot_metrics['accuracy']]
}
metrics_df = pd.DataFrame(metrics_data)

plt.figure(figsize=(8, 5))
sns.barplot(x='Method', y='Accuracy', data=metrics_df, palette='coolwarm')
plt.title('Accuracy Comparison: Zero-Shot vs. Few-Shot (Simulated)')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

# Add accuracy values on top of the bars
for index, row in metrics_df.iterrows():
    plt.text(index, row['Accuracy'] + 0.02, f"{row['Accuracy']:.4f}", color='black', ha="center")

plt.tight_layout()
plt.show()

## 9. Final Summary

Here's a summary of the performance of both classification approaches.

In [ ]:
print("\n--- Overall Performance Summary ---")
print(f"Zero-Shot Classification Accuracy: {zero_shot_metrics['accuracy']:.4f}")
print(f"Few-Shot Classification (Simulated) Accuracy: {few_shot_metrics['accuracy']:.4f}")

if few_shot_metrics['accuracy'] > zero_shot_metrics['accuracy']:
    print("\nConclusion: The Few-Shot Classification (Simulated) approach performed better in terms of accuracy.")
elif few_shot_metrics['accuracy'] < zero_shot_metrics['accuracy']:
    print("\nConclusion: The Zero-Shot Classification approach performed better in terms of accuracy.")
else:
    print("\nConclusion: Both Zero-Shot and Few-Shot Classification (Simulated) approaches performed similarly in terms of accuracy.")

print("\nNote: The 'Few-Shot' approach implemented here is a simulation by prepending examples to the input for a zero-shot model. True few-shot learning often involves generative models or meta-learning techniques.")